[Guide Index](README.md)

# 06.02 SPARQL queries, updates and rules as RDF

`starlayer.sparql` can represent SPARQL query text, SPARQL Update text and SRL (SPARQL Rules) rule sets as RDF graphs. Queries and updates use the `sqe:` ontology and rule sets use the `srl:` ontology (both specific to StarLayer). Each serializes text into a human-readable RDF syntax tree designed for storage, validation, viewing and editing. The RDF graph can be serialized back as plain text for query execution or rule inference.

The two ontologies share one way of ordering. Each ordered part of a tree (the projection, the elements of a WHERE clause, the body of a rule) carries its position as `tree:ord`, and a variable or an IRI is held in a `tree:Item`. Three functions work on all three kinds of text and find the kind themselves: `sparql_to_tree()`, `tree_to_sparql()` and `validate_sparql_tree()`.

Corresponding SHACL shapes graphs validate the `sqe:` and `srl:` graphs. The `sqe:` shapes incorporate `shui:` (SHACL UI) components that can be used in a SHACL-based editor of SPARQL queries.

See the [SPARQL guide](03.01-sparql.ipynb) for ordinary query operations and the [SRL (SPARQL Rules) guide](04.01-srl-sparql-rules.ipynb) for the rules language. This guide is specifically about treating a *query or rule set itself* as an RDF syntax tree that can be created, inspected, edited, and validated. [OWL Manchester Syntax as RDF](06.03-manchester-syntax-as-rdf.ipynb) does the same for Manchester Syntax, and [StarLayer Schemas and Ontologies](06.01-starlayer-schemas-and-ontologies.ipynb) lists all of the vocabularies.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). Run cells from top to bottom; later steps reuse variables from earlier ones.

In [2]:
from starlayer.graph import Namespace
from starlayer.graph import StarLayerGraph

EX = Namespace("http://example.org/")

## Index of functions

SPARQL syntax trees in `starlayer.sparql` have three functions. Each works on queries, updates and rule sets.

| Function | What it does | Discussed in |
|---|---|---|
| `sparql_to_tree()` | Parses SPARQL query text, SPARQL Update text or SRL rule set text into an RDF syntax tree graph: `sqe:` for a query or an update, `srl:` for a rule set. | sections 06.02.1 to 06.02.1.5, 06.02.2.2 and 06.02.5 |
| `tree_to_sparql()` | Renders one query, update or rule set of an RDF syntax tree graph back as text. | sections 06.02.1.2, 06.02.1.5, 06.02.2.2, 06.02.3 and 06.02.3.2 |
| `validate_sparql_tree()` | Validates a tree against the SHACL shapes of its ontology, checks that Oxigraph accepts each query, checks that a rule set is well formed and can be ordered, and returns a validation report. | sections 06.02.4 to 06.02.4.3 |

## 06.02.1 A graph, a query, and the query as RDF

`sparql_to_tree(text_query)` parses SPARQL text into an RDF graph, using the `sqe:` ontology. The graph is a syntax tree of the query with each part (e.g. SELECT, WHERE, FILTER, variables) reflected as an RDF element. Text that is not SPARQL cannot be parsed and raises an error (see section 06.02.5). A query that parses can still be invalid and get rejected when run. `validate_sparql_tree()` tests for validation errors (see section 06.02.4).

Below, one small data graph and one query are created to be used throughout this guide. The query is first run as ordinary SPARQL text, which finds that `ex:bob` knows `ex:carol`. It is then parsed into an RDF tree, and the tree is output as a graph.

Each ordered part of the tree (the projection, the elements of a WHERE clause, a function's arguments) carries its position as `tree:ord`. A variable or an IRI is held in a `tree:Item`.

In [3]:
from starlayer.sparql import sqe, sparql_to_tree
from starlayer.sparql.vocab import SPEXPR

g = StarLayerGraph()
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:bob ex:knows ex:carol ;
        ex:likes ex:dana .
""", format="turtle")

print("data graph:")
print(g.serialize(format="turtle", pretty=True))


# a simple SPARQL query
text_query = """PREFIX ex: <http://example.org/>
SELECT ?s ?o WHERE { ?s ex:knows ?o }"""

# run the query against the data graph, as ordinary SPARQL text
print("query result:")
for row in g.query(text_query):
    print(g.qname(row.s), g.qname(row.o))
print()

# the query is converted to an sqe syntax tree, written into the StarLayerGraph passed in; root is the base node of the query graph.
tree, root = sparql_to_tree(text_query, StarLayerGraph())

tree.bind("sqe", sqe.SQE)
tree.bind("spexpr", SPEXPR)
tree.bind("tree", "https://github.com/hidden-graph/starlayer/ns/tree#")

print(tree.serialize(pretty=True, only_used=True))

data graph:
@prefix ex: <http://example.org/> .

ex:bob ex:knows ex:carol ;
    ex:likes ex:dana .

query result:
ex:bob ex:carol

@prefix spexpr: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix sqe: <https://github.com/hidden-graph/starlayer/ns/sparql/query-edit#> .
@prefix tree: <https://github.com/hidden-graph/starlayer/ns/tree#> .

_:b0 a sqe:Query, sqe:SelectQuery ;
    sqe:prefixDecl [
        a sqe:PrefixDecl ;
        sqe:namespace <http://example.org/> ;
        sqe:prefixLabel "ex"^^spexpr:PlainString
    ] ;
    sqe:projection [
        a tree:Item ;
        tree:ord 1 ;
        tree:value "o"^^spexpr:Variable
    ], [
        a tree:Item ;
        tree:ord 0 ;
        tree:value "s"^^spexpr:Variable
    ] ;
    sqe:where [
        a sqe:TriplePattern ;
        sqe:object "o"^^spexpr:Variable ;
        sqe:predicate <http://example.org/knows> ;
        sqe:subject "s"^^spexpr:Variable ;
        tree:ord 0
    ] .



### 06.02.1.1 Giving the query a stable identity with `root=`

`sparql_to_tree()` gives the root node of the tree a fresh blank node by default, as seen above. Passing `root=` gives the query a real, caller-chosen identity instead, so it stays findable by that identity once a graph holds more than one encoded query.

Below, the same query is parsed with `root=` set to `q:bobKnows`, and the tree is printed with that IRI as its root.

In [4]:
from starlayer.graph import NamedNode


query_iri = NamedNode("http://example.org/queries#bobKnows")
named_tree, named_root = sparql_to_tree(text_query, root=query_iri)

print("root is the given IRI:", named_root.value == query_iri.value)

named_tree.bind("sqe", sqe.SQE)
named_tree.bind("spexpr", SPEXPR)
named_tree.bind("tree", "https://github.com/hidden-graph/starlayer/ns/tree#")
named_tree.bind("q", "http://example.org/queries#")
print(named_tree.serialize(pretty=True, only_used=True))

root is the given IRI: True
@prefix q: <http://example.org/queries#> .
@prefix spexpr: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix sqe: <https://github.com/hidden-graph/starlayer/ns/sparql/query-edit#> .
@prefix tree: <https://github.com/hidden-graph/starlayer/ns/tree#> .

q:bobKnows a sqe:Query, sqe:SelectQuery ;
    sqe:prefixDecl [
        a sqe:PrefixDecl ;
        sqe:namespace <http://example.org/> ;
        sqe:prefixLabel "ex"^^spexpr:PlainString
    ] ;
    sqe:projection [
        a tree:Item ;
        tree:ord 0 ;
        tree:value "s"^^spexpr:Variable
    ], [
        a tree:Item ;
        tree:ord 1 ;
        tree:value "o"^^spexpr:Variable
    ] ;
    sqe:where [
        a sqe:TriplePattern ;
        sqe:object "o"^^spexpr:Variable ;
        sqe:predicate <http://example.org/knows> ;
        sqe:subject "s"^^spexpr:Variable ;
        tree:ord 0
    ] .



### 06.02.1.2 Holding more than one query in a graph

A graph can hold more than one query. When `sparql_to_tree()` is given a graph, it adds the query's tree to that graph, and the root of each query identifies the different queries.

Below, a second query is parsed into the graph that already holds the first one, with its own root IRI. The graph then holds two queries. They are listed by their roots, and each is rendered back to SPARQL text through its root.

In [5]:
from starlayer.graph import RDF
from starlayer.sparql import tree_to_sparql

likes_text = """PREFIX ex: <http://example.org/>
SELECT ?s ?o WHERE { ?s ex:likes ?o }"""
likes_iri = NamedNode("http://example.org/queries#bobLikes")

# passing a graph that already holds a query adds the new query's tree to it
sparql_to_tree(likes_text, tree_graph=named_tree, root=likes_iri)

print("queries in the graph:", sorted(named_tree.qname(s) for s in named_tree.subjects(RDF.type, sqe.SQE.Query)))
for root_iri in (query_iri, likes_iri):
    print()
    print(named_tree.qname(root_iri))
    print(tree_to_sparql(named_tree, root_iri))

queries in the graph: ['q:bobKnows', 'q:bobLikes']

q:bobKnows
PREFIX ex: <http://example.org/>
SELECT ?s ?o
WHERE { ?s ex:knows ?o . }

q:bobLikes
PREFIX ex: <http://example.org/>
SELECT ?s ?o
WHERE { ?s ex:likes ?o . }


### 06.02.1.3 A rule set

`sparql_to_tree()` also reads SRL rule set text. It knows the text is a rule set when it starts with `RULE`, `DATA` or `IMPORTS` after any prologue. A rule set is run as a `srl.RuleSet`, the dataclass `srl.parse_ruleset()` returns (see the [SRL (SPARQL Rules) guide](04.01-srl-sparql-rules.ipynb) for the rules language itself).

Below, the same base graph and rule set as the [SRL (SPARQL Rules) guide](04.01-srl-sparql-rules.ipynb) are created - see that guide for what these rules mean; this one is only about representing them as RDF. The rule set text is parsed into a `RuleSet`.

The graphs here are `StarLayerGraph`, the Oxigraph-backed graph class. A fresh `StarLayerGraph` has no prefixes bound, so the cells bind the ones they print with.

In [1]:
from starlayer.graph import Namespace, RDF
from starlayer.graph import StarLayerGraph
from starlayer.sparql import srl

EX = Namespace("http://example.org/")

# StarLayerGraph is the Oxigraph-backed graph class (see the Oxigraph-backed graph guide)
base = StarLayerGraph()
base.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:frontend a ex:Component ; ex:dependsOn ex:app .
    ex:app      a ex:Component ; ex:dependsOn ex:db .
    ex:db       a ex:Component ; ex:hasVulnerability ex:vuln1 .
    ex:logger   a ex:Component .

    ex:vuln1 ex:severity 9.1 .
""", format="turtle")
base.bind("ex", EX)  # a fresh StarLayerGraph has no prefixes of its own; bind what is needed

rules_text = """
PREFIX ex: <http://example.org/>

RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }

RULE { ?x ex:status ex:criticallyExposed }
WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= 9.0) }

RULE { ?x ex:status ex:safeToDeploy }
WHERE { ?x a ex:Component . NOT { ?x ex:status ex:criticallyExposed } }
"""

ruleset = srl.parse_ruleset(rules_text)
print(f"{len(ruleset.rules)} rules parsed")

4 rules parsed


### 06.02.1.4 The rule set as RDF

`sparql_to_tree()` encodes rule set text as RDF under the `srl:` namespace - the same tree-RDF pattern `sqe:` uses for SPARQL queries (section 06.02.1). It encodes into the graph passed in (here an empty `StarLayerGraph`, so the tree lives in Oxigraph) and returns `(graph, root)` - `root` is the `srl:RuleSet` node's own identity, needed later to decode a specific rule set back out of the graph. `srl.ruleset_to_tree()` does the same for a `RuleSet` that is already parsed.

Each rule, and each `NOT`, carries the spec's flag `evalground` as `srl:ruleEvalground` or `srl:negationEvalground`: `true` means the patterns match the evaluation graph (a plain `WHERE` or `NOT`), `false` means they match only the base graph (`WHERE GROUND` or `NOT GROUND`; see the [SRL (SPARQL Rules) guide](04.01-srl-sparql-rules.ipynb)).

Below, the rule set text is read into an `srl:` tree, and the tree is printed.

In [2]:
srl_graph, srl_root = sparql_to_tree(rules_text, StarLayerGraph())

srl_graph.bind("srl", srl.SRL)
srl_graph.bind("spexpr", SPEXPR)
srl_graph.bind("tree", "https://github.com/hidden-graph/starlayer/ns/tree#")
srl_graph.bind("ex", EX)

print("ruleset as graph:")
print(srl_graph.serialize(format="turtle", pretty=True))

ruleset as graph:
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix spexpr: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix srl: <https://github.com/hidden-graph/starlayer/ns/sparql/srl#> .
@prefix tree: <https://github.com/hidden-graph/starlayer/ns/tree#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

_:b0 a srl:RuleSet ;
    srl:rules [
        a srl:Rule ;
        srl:body [
            a srl:TriplePattern ;
            srl:object "v"^^spexpr:Variable ;
            srl:predicate ex:exposedTo ;
            srl:subject "y"^^spexpr:Variable ;
            tree:ord 1
        ], [
            a srl:TriplePattern ;
            srl:object "y"^^spexpr:Variable ;
            srl:predicate ex:dependsOn ;
            srl:subject "x"^^spexpr:Variable ;
            tree:ord 0
        ] ;
        srl:head [
            a srl:TriplePattern ;
            srl:object "v"^^spexpr:Variable ;
            srl:predicate ex:

### 06.02.1.5 Update text and empty text

`sparql_to_tree()` also reads SPARQL Update text. An update gives an `sqe:UpdateRequest` tree, not a query tree. Text that holds only a prologue (`PREFIX`, `BASE`, `VERSION`, comments) or nothing at all is an update request with no operation: it loads as an empty `sqe:UpdateRequest`, and `tree_to_sparql()` writes back its prologue, or an empty string.

Below, an update is parsed into a tree and written back as text. Then empty text is parsed, and the tree is written back.

In [7]:
update_text = """PREFIX ex: <http://example.org/>
INSERT DATA { ex:bob ex:knows ex:erin }"""
update_tree, update_root = sparql_to_tree(update_text)
print("class of the root:", [o.value.rsplit("#", 1)[-1] for o in update_tree.objects(update_root, RDF.type)])
print(tree_to_sparql(update_tree, update_root))
print()

empty_tree, empty_root = sparql_to_tree("")
print("class of the root:", [o.value.rsplit("#", 1)[-1] for o in empty_tree.objects(empty_root, RDF.type)])
print("text of the empty request:", repr(tree_to_sparql(empty_tree, empty_root)))

class of the root: ['UpdateRequest']
PREFIX ex: <http://example.org/>
INSERT DATA { ex:bob ex:knows ex:erin . }

class of the root: ['UpdateRequest']
text of the empty request: ''


## 06.02.2 Edit the tree by editing the RDF

A query tree is an ordinary RDF graph, so it can be edited with standard graph operations. Edits to the graph structure change the query the tree represents. Annotations that the query does not use, such as a label or a comment, are ignored when the tree is turned back into text (section 06.02.3).

Below, a triple pattern of the tree is found by its predicate `ex:knows` and changed to `ex:likes`. A plain label is added to the query as a whole, and a comment is attached to the query's own `rdf:type` statement as an RDF 1.2 reification, to record why the predicate was changed. The edited tree is printed as a graph.

In [6]:
from starlayer.graph import BlankNode, Literal, RDF, RDFS

# the triple pattern is located and its predicate replaced.
for s, p, o in list(tree.triples((None, sqe.SQE.predicate, EX.knows))):
    tree.remove((s, p, o))
    tree.add((s, p, EX.likes))
    pattern_node = s

# a plain annotation on the query as a whole
tree.add((root, RDFS.label, Literal("bob's likes, not knows")))

# a real RDF 1.2 reification: the query's own rdf:type assertion is annotated
# to record why the predicate of its triple pattern was changed
reifier = BlankNode()
tree.add((reifier, RDFS.comment, Literal("predicate changed from ex:knows to ex:likes")))
tree.add_reification(reifier, (root, RDF.type, sqe.SQE.SelectQuery))

tree.bind("rdfs", RDFS)
print(tree.serialize(pretty=True, only_used=True))

@version "1.2" .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix spexpr: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix sqe: <https://github.com/hidden-graph/starlayer/ns/sparql/query-edit#> .
@prefix tree: <https://github.com/hidden-graph/starlayer/ns/tree#> .

_:b0 a sqe:Query, sqe:SelectQuery {| rdfs:comment "predicate changed from ex:knows to ex:likes" |} ;
    rdfs:label "bob's likes, not knows" ;
    sqe:prefixDecl [
        a sqe:PrefixDecl ;
        sqe:namespace <http://example.org/> ;
        sqe:prefixLabel "ex"^^spexpr:PlainString
    ] ;
    sqe:projection [
        a tree:Item ;
        tree:ord 0 ;
        tree:value "s"^^spexpr:Variable
    ], [
        a tree:Item ;
        tree:ord 1 ;
        tree:value "o"^^spexpr:Variable
    ] ;
    sqe:where [
        a sqe:TriplePattern ;
        sqe:object "o"^^spexpr:Variable ;
        sqe:predicate <http://example.org/likes> ;
        sqe:subject "s"^^spexpr:Variable ;
        tree:ord 0

### 06.02.2.1 Edit a rule set by editing the RDF

The rule set graph is an ordinary `StarLayerGraph` and can be edited as one (`set(triple)` replaces the value a subject has for a predicate; `remove` followed by `add` does the same for one value).

Below, rule 2's own body - `?x ex:dependsOn ?y . ?y ex:exposedTo ?v` - is edited so the dependency edge it walks is `ex:reliesOn` instead of `ex:dependsOn`, directly as RDF graph surgery, no text-level find/replace anywhere.

In [3]:
# a triple pattern's own predicate is located and replaced
for s in list(srl_graph.subjects(srl.SRL.predicate, EX.dependsOn)):
    srl_graph.remove((s, srl.SRL.predicate, None))
    srl_graph.add((s, srl.SRL.predicate, EX.reliesOn))

print("edited: rule 2's body now walks ex:reliesOn, not ex:dependsOn")

edited: rule 2's body now walks ex:reliesOn, not ex:dependsOn


### 06.02.2.2 Edit the order of the parts

The ordered parts of a tree are plain triples, and each child carries its position as `tree:ord`. Editing the order is therefore editing those numbers. The helpers of `starlayer.graph.ordinals` do it: `append()`, `insert_at()`, `remove_at()` and `replace_at()` each edit one ordered sequence, named by its parent node and property, and keep the `tree:ord` numbers and the `tree:Item` wrappers right.

Below, a fresh tree of the query from section 06.02.1 is made, and its projection `?s ?o` is changed to `?o ?s`. `remove_at()` takes out the first variable and `append()` puts it last. The tree is then written back as text.

In [10]:
from starlayer.graph import ordinals

order_tree, order_root = sparql_to_tree(text_query)

# take out the first projected variable and add it again at the end
first = ordinals.remove_at(order_tree, order_root, sqe.SQE.projection, 0)
ordinals.append(order_tree, order_root, sqe.SQE.projection, first)

print(tree_to_sparql(order_tree, order_root))

PREFIX ex: <http://example.org/>
SELECT ?o ?s
WHERE { ?s ex:knows ?o . }


## 06.02.3 Decode back into SPARQL text, and run it

`tree_to_sparql()` renders an RDF tree back as SPARQL text, either to run it or to export it. A `StarLayerGraph` runs queries by text, so a tree is run as a query by rendering it to text first. Edits to the tree that do not affect what the query means are ignored. Annotations are not preserved in the text form: to retain them, store the query as a graph, not as text.

Below, the edited tree is rendered to text and the text is printed. The text is then run against the same data graph from section 06.02.1. The result changes from `(bob, carol)` to `(bob, dana)`, reflecting the edit made in section 06.02.2.

In [7]:
# the rdf graph becomes SPARQL text
modified_text = tree_to_sparql(tree, root)
print(modified_text)
print()

# the text is executed by Oxigraph
result = g.query(modified_text)

for row in result:
    print(g.qname(row.s), g.qname(row.o))

PREFIX ex: <http://example.org/>
SELECT ?s ?o
WHERE { ?s ex:likes ?o . }

ex:bob ex:dana


### 06.02.3.1 Decode a rule set, and run it

`tree_to_ruleset()` turns a rule set tree back into a real `RuleSet`, and `infer()` runs it.

Below, the edited RDF from section 06.02.2.1 is decoded and both rule sets are run against the *same* base graph. The base data's dependency chain is `frontend -> app -> db`, with the only vulnerability on `db` itself. Running both proves the edit is semantic: breaking the `ex:dependsOn` propagation edge stops `app`/`frontend` from inheriting `db`'s exposure, so only `db` stays `criticallyExposed`.

In [4]:
edited_ruleset = srl.tree_to_ruleset(srl_graph, srl_root)

original_inferred = ruleset.infer(base)
edited_inferred = edited_ruleset.infer(base)

print("original criticallyExposed:", sorted(base.qname(s) for s in original_inferred.subjects(EX.status, EX.criticallyExposed)))
print("edited criticallyExposed:  ", sorted(base.qname(s) for s in edited_inferred.subjects(EX.status, EX.criticallyExposed)))

original criticallyExposed: ['ex:app', 'ex:db', 'ex:frontend']
edited criticallyExposed:   ['ex:db']


### 06.02.3.2 Export the modified rule set back to text

`ruleset_to_text()` renders the edited `RuleSet` back to SRL text. Unlike `tree_to_sparql()`'s own `PREFIX` preservation for queries, the parsed SRL syntax tree only ever holds fully-resolved absolute IRIs - no original `PREFIX`/`BASE` declaration survives to render back. Passing `namespace_manager=` abbreviates the output; it accepts a `StarLayerGraph` (here the base graph, so its `ex:` binding is used), a prefix-to-namespace `dict` or an rdflib `NamespaceManager`. Only prefixes that were bound are used, which is why `rdf:type` stays a full IRI in the output below (the base graph has no `rdf:` binding); without the argument every IRI renders in full.

`tree_to_sparql()` writes a rule set tree to text as well, without the `RuleSet` step, and without prefixes.

Below, the edited rule set is written with `ruleset_to_text()` and the `ex:` prefix of the base graph. Then `tree_to_sparql()` writes the same rule set from the tree, with every IRI in full.

In [5]:
print(srl.ruleset_to_text(edited_ruleset, namespace_manager=base))

RULE { ?x ex:exposedTo ?v . } WHERE { ?x ex:hasVulnerability ?v . }
RULE { ?x ex:exposedTo ?v . } WHERE { ?x ex:reliesOn ?y . ?y ex:exposedTo ?v . }
RULE { ?x ex:status ex:criticallyExposed . } WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= "9"^^<http://www.w3.org/2001/XMLSchema#decimal>) }
RULE { ?x ex:status ex:safeToDeploy . } WHERE { ?x <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> ex:Component . NOT { ?x ex:status ex:criticallyExposed . } }


In [14]:
print(tree_to_sparql(srl_graph, srl_root))

RULE { ?x <http://example.org/exposedTo> ?v . } WHERE { ?x <http://example.org/hasVulnerability> ?v . }
RULE { ?x <http://example.org/exposedTo> ?v . } WHERE { ?x <http://example.org/reliesOn> ?y . ?y <http://example.org/exposedTo> ?v . }
RULE { ?x <http://example.org/status> <http://example.org/criticallyExposed> . } WHERE { ?x <http://example.org/exposedTo> ?v . ?v <http://example.org/severity> ?s . FILTER(?s >= "9"^^<http://www.w3.org/2001/XMLSchema#decimal>) }
RULE { ?x <http://example.org/status> <http://example.org/safeToDeploy> . } WHERE { ?x <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://example.org/Component> . NOT { ?x <http://example.org/status> <http://example.org/criticallyExposed> . } }


## 06.02.4 Validate

`validate_sparql_tree()` checks a tree against the SHACL shapes of its ontology, `sqe:` or `srl:`, and adds the checks the shapes cannot state. For an `sqe:` tree, if the tree conforms, it also renders each query in the tree to text and checks that Oxigraph accepts it. This step catches errors that the SHACL shapes cannot, for example a call to a function that Oxigraph does not support. The tree satisfies the shapes in that case, but the report fails, with a violation on the query's root node.

The shapes used for validation are the registered `sqe_shacl` shapes graph, the file `starlayer/starontology/sqe/sqe_shapes.ttl` of the package (see section 06.04.3 of the [SHACL library registry guide](06.04-shacl-library-registry.ipynb)).

Below, `validate_sparql_tree()` checks the edited query tree from section 06.02.2, and it conforms.

In [8]:
from starlayer.sparql import validate_sparql_tree

# validate the edited syntax tree from above.
report = validate_sparql_tree(tree)
print("conforms:", report.conforms)

conforms: True


Editing the tree can make the query invalid. Below, the triple pattern's own required `sqe:predicate` is removed, and `validate_sparql_tree()` reports the violation.

In [9]:
# break the tree: remove the triple pattern's own required predicate
pattern_node = next(tree.subjects(RDF.type, sqe.SQE.TriplePattern))
tree.remove((pattern_node, sqe.SQE.predicate, None))

broken_report = validate_sparql_tree(tree)
print("conforms:", broken_report.conforms)
print(broken_report.report_text)

conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:MinCountConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:maxCount 1 ; sh:message "sqe:TriplePattern must have exactly one sqe:predicate, an IRI/Variable/property path" ; sh:minCount 1 ; sh:node sqe:PredicateOrVariableOrPathShape ; sh:path sqe:predicate ]
	Focus Node: [ rdf:type sqe:TriplePattern ; sqe:object "o"^^spexpr:Variable ; sqe:subject "s"^^spexpr:Variable ; tree:ord 0 ]
	Result Path: sqe:predicate
	Message: sqe:TriplePattern must have exactly one sqe:predicate, an IRI/Variable/property path



For a rule set, `parse_ruleset()` and `sparql_to_tree()` accept any text the SRL grammar accepts, so a rule set with problems can still be loaded, inspected and fixed. They raise only for text that is not SRL (section 06.02.5). `validate_sparql_tree()` reports the problems in the rule set's syntax tree. It reports the §4.2 well-formedness conditions (an unbound head, filter or assignment variable, a `SET(...)` reusing an already-bound variable, an aggregate used inside `FILTER` or `SET`), the structure checked by the `srl:` SHACL shapes, and a rule set that cannot be put in an evaluation order. Each §4.2 problem is a result on the element at fault, with a message that names the variable. A rule set that has `IMPORTS` that are not resolved gets a warning, which does not make the report fail.

`validate_sparql_tree()` is also needed whenever a tree is edited by hand, since those edits (like section 06.02.2.1's above) happen outside of `parse_ruleset()`. A rule set that breaks §4.2 is also refused when it is run: `infer()` raises a `FormatError` (kind `"srl"`).

Below, `validate_sparql_tree()` checks the edited rule set tree from section 06.02.2.1, and it conforms.

In [6]:
from starlayer.sparql import validate_sparql_tree

# validate the edited syntax tree from section 06.02.2.1 above
report = validate_sparql_tree(srl_graph)
print("conforms:", report.conforms)

conforms: True


Editing the tree can also make it invalid. Removing a `srl:TriplePattern`'s own required `srl:predicate` is one example - `validate_sparql_tree()` catches it:

In [7]:
# break the tree: remove a triple pattern's own required predicate
pattern_node = next(srl_graph.subjects(RDF.type, srl.SRL.TriplePattern))
srl_graph.remove((pattern_node, srl.SRL.predicate, None))

broken_report = validate_sparql_tree(srl_graph)
print("conforms:", broken_report.conforms)
print(broken_report.report_text)

conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:MinCountConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:maxCount 1 ; sh:message "srl:TriplePattern must have exactly one srl:predicate, an IRI/Variable" ; sh:minCount 1 ; sh:node srl:PredicateOrVariableShape ; sh:path srl:predicate ]
	Focus Node: [ <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> srl:BodyElement, srl:GroupBodyElement, srl:NegationBodyElement, srl:TriplePattern ; srl:object ex:criticallyExposed ; srl:subject "x"^^spexpr:Variable ; tree:ord 0 ]
	Result Path: srl:predicate
	Message: srl:TriplePattern must have exactly one srl:predicate, an IRI/Variable



### 06.02.4.1 A query that's valid SPARQL, but fails SHACL validation

A query can be valid SPARQL and still be wrong. For example `SELECT ?s ?z WHERE { ?s ex:knows ?o }` parses and runs as SPARQL without an error. `?z` is never bound by the pattern, so it comes back unbound in every result row. SPARQL itself has no objection to this, but it almost always signals a typo or an incomplete edit: an "output" (`?z` projected) with no matching "input" (nothing in `?s ex:knows ?o` ever binds it). `sqe:SelectQueryShape` carries a SHACL constraint able to catch errors of this nature.

Below, the query runs as ordinary SPARQL and `?z` comes back unbound. The same query is parsed into a tree, which loads as usual, and `validate_sparql_tree()` reports the problem. The validation report is long because it includes the constraint that found the problem: the SPARQL query under "Source Constraint" is the shape's own check over the tree, not the query being validated.

In [12]:
bad_query = """PREFIX ex: <http://example.org/>
SELECT ?s ?z WHERE { ?s ex:knows ?o }"""

# still valid SPARQL - runs fine, ?z just comes back unbound
print("query execution:")
for row in g.query(bad_query):
    print(g.qname(row.s), row.z)
print()

# the query loads as a tree; validate_sparql_tree() reports what is wrong with it
bad_tree, bad_root = sparql_to_tree(bad_query)
bad_report = validate_sparql_tree(bad_tree)
print("conforms:", bad_report.conforms)
print(bad_report.report_text)

query execution:
ex:bob None

conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:SPARQLConstraintComponent:
	Severity: sh:Violation
	Source Shape: sqe:SelectQueryShape
	Focus Node: [ rdf:type sqe:Query, sqe:SelectQuery ; sqe:prefixDecl [ rdf:type sqe:PrefixDecl ; sqe:namespace <http://example.org/> ; sqe:prefixLabel "ex"^^spexpr:PlainString ] ; sqe:projection [ rdf:type tree:Item ; tree:ord 0 ; tree:value "s"^^spexpr:Variable ], [ rdf:type tree:Item ; tree:ord 1 ; tree:value "z"^^spexpr:Variable ] ; sqe:where [ rdf:type sqe:TriplePattern ; sqe:object "o"^^spexpr:Variable ; sqe:predicate <http://example.org/knows> ; sqe:subject "s"^^spexpr:Variable ; tree:ord 0 ] ]
	Value Node: [ rdf:type sqe:Query, sqe:SelectQuery ; sqe:prefixDecl [ rdf:type sqe:PrefixDecl ; sqe:namespace <http://example.org/> ; sqe:prefixLabel "ex"^^spexpr:PlainString ] ; sqe:projection [ rdf:type tree:Item ; tree:ord 0 ; tree:value "s"^^spexpr:Variable ], [ rdf:type tree:Item ; 

### 06.02.4.2 A rule set that's well-formed SPARQL-wise, but fails SRL's own §4.2 check

`RULE { ?x ex:status ?z } WHERE { ?x a ex:Component . }` parses as a perfectly ordinary rule - but `?z` in the head is never bound anywhere in the body. §4.2's well-formedness condition requires every head variable to appear in the body's own bound-variable set; `?z` doesn't, so this signals a typo or an incomplete rule, the same "output with no matching input" shape as `sqe:`'s own unbound-projected-variable check in section 06.02.4.1.

Below, the text is parsed into a rule set, which loads as usual. `validate_sparql_tree()` then reports the unbound head variable, and the message of each result is printed. Running the rule set raises a `FormatError`, so the typo does not give a wrong answer later.

In [8]:
from starlayer.errors import FormatError

bad_rules = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:status ?z } WHERE { ?x a ex:Component . }
"""

# the rule set loads; validate_sparql_tree() reports what is wrong with its tree
bad_ruleset = srl.parse_ruleset(bad_rules)
bad_tree, bad_root = srl.ruleset_to_tree(bad_ruleset)
bad_report = validate_sparql_tree(bad_tree)
print("conforms:", bad_report.conforms)
# the report holds one result for each problem; each message names the variable
for row in bad_report.report_graph.query("SELECT ?message WHERE { ?result <http://www.w3.org/ns/shacl#resultMessage> ?message }"):
    print(row.message.value)

# running it is refused
try:
    bad_ruleset.infer(StarLayerGraph())
except FormatError as exc:
    print("infer() raised:", exc)

conforms: False
The rule head uses the variable z, which the rule body never binds.
infer() raised: SRL: rule set is not well-formed per §4.2: unbound_head_variable (?z in rule head)


### 06.02.4.3 A rule set that cannot be ordered, and imports that are not resolved

Rules can depend on each other, and a negation makes a dependency that must not run in a circle: a rule set whose rules negate each other's results has no well-defined outcome, and `validate_sparql_tree()` reports it as a violation on the rule set (section 4.4 of the SRL specification). A rule set with an `IMPORTS` entry that is not resolved is checked without the imported rules, so the report holds a warning that says so. A warning does not make the report fail.

Below, two rules that negate each other's heads are validated, and then a rule set with an unresolved import. The severity and the message of each result are printed.

In [21]:
circular = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:p ex:a } WHERE { ?x a ex:C . NOT { ?x ex:q ex:a } }
RULE { ?x ex:q ex:a } WHERE { ?x a ex:C . NOT { ?x ex:p ex:a } }
"""
imports = """
PREFIX ex: <http://example.org/>
IMPORTS <http://example.org/other>
RULE { ?x ex:b ?y } WHERE { ?x ex:a ?y . }
"""

for name, rules in (("circular", circular), ("imports", imports)):
    rules_tree, _ = sparql_to_tree(rules)
    result_report = validate_sparql_tree(rules_tree)
    print(name, "- conforms:", result_report.conforms)
    for row in result_report.report_graph.query(
        "SELECT ?severity ?message WHERE { ?r <http://www.w3.org/ns/shacl#resultSeverity> ?severity ; <http://www.w3.org/ns/shacl#resultMessage> ?message }"
    ):
        print("  ", row.severity.value.rsplit("#", 1)[-1], "-", row.message.value)

circular - conforms: False
   Violation - SRL: stratification condition violated - a recursive dependency involves a closed edge, so this rule set has no well-defined evaluation outcome per §4.4.1
imports - conforms: True
   Warning - The rule set has an IMPORTS entry http://example.org/other that is not resolved, so the check that the rules can be ordered covers only the rules in this document; an imported rule can add a dependency that makes them unorderable. Resolve the imports (srl.resolve_imports(), or imports_loader=) and validate again.


## 06.02.5 Text that is not SPARQL or SRL

`sparql_to_tree()` finds the kind of text itself. It tries the SPARQL query grammar, then the update grammar, then the SRL grammar when the text starts with `RULE`, `DATA` or `IMPORTS`. Text that no grammar reads raises a `FormatError`. Its `kind` is `"srl"` when the text starts as a rule set and the SRL grammar read furthest into it, and `"sparql"` otherwise. The error holds the `line` and `col` where the grammar stopped.

Below, plain text that is not SPARQL is refused, and then a rule set that stops before it is complete. The kind, the line and the column of each error are printed, with its message.

In [22]:
from starlayer.errors import FormatError

for not_text in ("this is not sparql", "PREFIX ex: <http://example.org/>\nRULE { ?x ex:p ?y } WHERE { ?x ex:q"):
    try:
        sparql_to_tree(not_text)
    except FormatError as exc:
        print(f"kind {exc.kind!r}, line {exc.line}, column {exc.col}:", exc)

kind 'sparql', line 1, column 1: SPARQL syntax error at line 1, column 1: expected SELECT, ASK, CONSTRUCT or DESCRIBE (found 'this')
kind 'srl', line 2, column 36: SPARQL syntax error at line 2, column 36: expected an RDF term or variable (found end of input)
